# Lab 3: Build a Policy Search Store

**BADM 6121: Applied AI Lab**

This notebook does the **index once** half of a RAG system. It uploads the MBA policy PDFs to **Gemini API File Search**, which splits them into chunks, embeds each chunk, and stores the vectors. Then it asks a question and shows which chunks the answer came from.

The app does the **every question** half. It sends a question to Gemini with the same store attached.

**Before you start:** click **File > Save a copy in Drive**.

### 1. Install the Gemini library

In [ ]:
# It comes installed on Google Colab.
# But if you were running in another environment, uncomment the below

# !pip install -q -U google-genai

### 2. Connect with your API key

1. Get a key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey). Note which **project** it belongs to: the store you create lives in that project, and the app must use a key from the same project.
2. In Colab, click the key icon on the left (**Secrets**), then **Add new secret**.
3. Name: `AI_STUDIO_API_KEY`. Value: paste the key. Turn on **Notebook access**.

Never paste a key into a cell.

In [1]:
from google import genai
from google.genai import types
from google.colab import userdata
import time

client = genai.Client(api_key=userdata.get("AI_STUDIO_API_KEY"))
MODEL = "gemini-3.5-flash"

### 3. Upload the policy PDFs to Colab

Run the cell and choose the three PDFs:
- MBA Student Handbook 2026-2027
- MBA Student Syllabus Policies 2026-2027
- Artificial Intelligence (AI) Use Policy, MBA Program

In [2]:
from google.colab import files

uploaded = files.upload()
list(uploaded)

Saving Artificial Intelligence (AI) Use Policy- MBA Program.pdf to Artificial Intelligence (AI) Use Policy- MBA Program.pdf
Saving MBA Student Handbook 2026-2027 v2.docx.pdf to MBA Student Handbook 2026-2027 v2.docx.pdf
Saving MBA Student Syllabus Policies 2026-2027.pdf to MBA Student Syllabus Policies 2026-2027.pdf


['Artificial Intelligence (AI) Use Policy- MBA Program.pdf',
 'MBA Student Handbook 2026-2027 v2.docx.pdf',
 'MBA Student Syllabus Policies 2026-2027.pdf']

### 4. Create a File Search store

Finds the mba-policies store, or creates it if it doesn’t exist yet. Copy the printed name: the app needs it.

In [3]:
STORE_NAME = None
for s in client.file_search_stores.list():
    if s.display_name == "mba-policies":
        STORE_NAME = s.name

if STORE_NAME is None:
    STORE_NAME = client.file_search_stores.create(config={"display_name": "mba-policies"}).name
    print("Created:", STORE_NAME)
else:
    print("Reusing:", STORE_NAME)

Created: fileSearchStores/mbapolicies-ldjpw9byydqw


### 5. Add the PDFs to the store

For each PDF, File Search extracts the text, splits it into chunks of up to 200 tokens (with 20 tokens of overlap between neighbors), and embeds every chunk. This is the slow step: expect a minute or two per document.

The cell repeats the same steps for each file you uploaded.

In [4]:
# checks the name of documents already loaded into the database, if any
already_indexed = [doc.display_name for doc in client.file_search_stores.documents.list(parent=STORE_NAME)]

# imports every file you uploaded to the database if it doesn't already exist
for filename in uploaded:
    if filename in already_indexed:
        print("Already indexed:", filename)
        continue
    operation = client.file_search_stores.upload_to_file_search_store(
        file=filename,
        file_search_store_name=STORE_NAME,
        config={
            "display_name": filename,
            "chunking_config": {
                "white_space_config": {
                    "max_tokens_per_chunk": 200,
                    "max_overlap_tokens": 20,
                }
            },
        },
    )
    while not operation.done:
        time.sleep(5)
        operation = client.operations.get(operation)
    print("Indexed:", filename)

Indexed: Artificial Intelligence (AI) Use Policy- MBA Program.pdf
Indexed: MBA Student Handbook 2026-2027 v2.docx.pdf
Indexed: MBA Student Syllabus Policies 2026-2027.pdf


Check what is in the store:

In [5]:
for doc in client.file_search_stores.documents.list(parent=STORE_NAME):
    print(doc.display_name, "|", doc.state)

Artificial Intelligence (AI) Use Policy- MBA Program.pdf | DocumentState.STATE_ACTIVE
MBA Student Handbook 2026-2027 v2.docx.pdf | DocumentState.STATE_ACTIVE
MBA Student Syllabus Policies 2026-2027.pdf | DocumentState.STATE_ACTIVE


### 6. Ask a question

One request does the whole **every question** half: Gemini embeds the question, finds the closest chunks in the store, puts them in the model's context, and writes the answer.

In [6]:
question = "How many absences are allowed before a student fails a course?"

response = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        tools=[types.Tool(file_search=types.FileSearch(file_search_store_names=[STORE_NAME]))]
    ),
)
print(response.text)

The allowance for absences depends on the course and program level:

### **Full-Time MBA First-Year Core Courses**
* **Maximum Absence Cap (Automatic Failure):** Missing **more than 50%** of scheduled class sessions will result in an automatic failing grade in the course. This rule applies regardless of whether the absences are excused or unexcused.
* **Excused Absences:** Instructors may excuse up to **two (2) absences** without academic penalty. However, even excused absences count toward the 50% maximum threshold.

### **All Other MBA Courses**
*(Electives, Intensive Learning Experiences (ILEs), and core courses for Part-time/Flex, Online/Flex Online, and 2nd-year Full-time students)*
* Attendance requirements for these courses are established by **individual faculty members**. 
* Faculty may adopt the 1st-Year Core Policy (50% cap) or enforce their own course-specific expectations. You should review the individual course syllabus on Canvas or speak directly with your instructor reg

### 7. Check the evidence

The response also lists the chunks the answer was built from: which document, which page, and the text. Read them. Does each claim in the answer appear in a chunk?

In [7]:
metadata = response.candidates[0].grounding_metadata

for chunk in metadata.grounding_chunks:
    context = chunk.retrieved_context
    print(context.title, "| page", context.page_number)
    print(context.text[:400])
    print("---")

MBA Student Syllabus Policies 2026-2027.pdf | page 5
professional development and
personal commitments. Our attendance policies are designed to maintain academic rigor while providing
reasonable flexibility for legitimate career opportunities and personal needs.
Full-time MBA First Year Core Courses (effective Module 1, Fall 2026)
The following policy applies to all students enrolled in 1st year Full-time MBA foundational and custom
core courses:
•

---
MBA Student Handbook 2026-2027 v2.docx.pdf | page 15
anticipate missing a class. In the
case of unscheduled or emergency absences, contact faculty as soon as possible, ideally before
class.
Excused Absences: Instructors may excuse up to two absences without academic penalty.
Student Responsibility: Students are responsible for managing their attendance throughout the
term. Those who know in advance that they will miss class for approved religious ob
---


If this cell fails with an error about `None`, no chunks were retrieved: the model answered without the documents.

### 8. Test it

Change `question` in step 6 and rerun steps 6 and 7. Try:
- A fact you can look up yourself in the PDFs
- A question that needs two different sections
- The same question in different words than the policy uses
- A question the policies do not cover. A good answer says so.

### 9. Housekeeping

List every store in your project:

In [8]:
for s in client.file_search_stores.list():
    print(s.name, "|", s.display_name)

fileSearchStores/hoyarag-georgetown-mba-stor-p5mo7dd1ctjv | HoyaRAG Georgetown MBA Store
fileSearchStores/hoyarag-georgetown-mba-stor-gaazt4m918no | HoyaRAG Georgetown MBA Store
fileSearchStores/hoyarag-georgetown-mba-stor-6mo6onmcx1i1 | HoyaRAG Georgetown MBA Store
fileSearchStores/mbapolicies-ldjpw9byydqw | mba-policies


Delete a store you no longer need (remove the `#` to run):

In [ ]:
# client.file_search_stores.delete(name=STORE_NAME, config={"force": True})